# Source-Grounded Graph Food Substitution Recommender — Kaggle Edition
#
Extended, Kaggle-friendly rebuild of the `source_grounded_graph_project_scripts`
training pipeline. Self-contained: no need to upload the original project's
`.py`/`.yaml` files, just the three processed data files described in the
"HOW TO USE" note below.
#
**What changed relative to the original `run_source_grounded_graph_experiment.py`:**
1. Pareto-frontier ranking removed entirely (function, diagnostic column, and
   coverage metric) and replaced with a continuous objective trade-off analysis.
2. Bayesian hyperparameter + architecture search with Optuna (TPE sampler,
   median pruning), searching width/depth/dropout *and* real architectural
   choices: residual connections, activation, message aggregation, head size.
3. Fixed an unused 10% "gap" in the original standard split — it's now a
   proper validation set used for HPO, keeping the test set leakage-free.
4. New analyses on the final trained model: relation ablation study,
   bootstrap confidence intervals, behavior-score calibration, cold-start
   fairness segmentation by SNAP/poverty band, embedding-space diagnostics,
   and permutation feature importance.
#
**HOW TO USE ON KAGGLE**
1. Run the local project pipeline through step 6
   (`build_food_relations_database.py`) to produce three files:
   `food_relations.sqlite`, `food_catalog.json`, `substitution_pairs.json`.
2. On kaggle.com -> Datasets -> New Dataset -> upload those three files.
3. In this notebook: Add Data -> select that dataset.
4. Settings (right sidebar) -> Accelerator -> GPU T4 x2 (optional but faster).
   Settings -> Internet -> On (needed once, to `pip install optuna`).
5. Run all cells.

In [103]:
# ============================================================================
# 0. INSTALL / IMPORTS
# ============================================================================
import subprocess
import sys

try:
    import optuna  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "optuna"], check=True)
    import optuna  # noqa: F401

import glob
import json
import math
import random
import sqlite3
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Optional
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="white")

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "savefig.edgecolor": "white",
    "axes.grid": False,
})
import numpy as np
import pandas as pd
import torch
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
from sklearn.decomposition import PCA
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    log_loss,
    mean_absolute_error,
    mean_squared_error,
    roc_auc_score,
)
from sklearn.metrics.pairwise import cosine_similarity
from torch import nn
from torch.nn import functional as F

warnings.filterwarnings("ignore", category=FutureWarning)


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cuda


In [104]:
# ============================================================================
# 1. CONFIG — everything you're likely to want to tweak lives here
# ============================================================================

# Set True the first time you run this on a new dataset: cuts trial/epoch/
# bootstrap counts down so the *entire* notebook finishes in a couple of
# minutes, just to confirm data loads and every section runs end to end.
# Set False for the real run.
QUICK_TEST_MODE = False

# If auto-discovery under /kaggle/input fails (e.g. running locally instead
# of on Kaggle), set this to the folder containing the three processed files.
MANUAL_DATA_DIR: Optional[str] = None

OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./kaggle_working")
for sub in ["checkpoints", "figures", "tables"]:
    (OUTPUT_DIR / sub).mkdir(parents=True, exist_ok=True)

SEED = 2026
N_OPTUNA_TRIALS = 6 if QUICK_TEST_MODE else 30
HPO_EPOCHS = 8 if QUICK_TEST_MODE else 40
FINAL_EPOCHS = 75 if QUICK_TEST_MODE else 150
N_BOOTSTRAP = 20 if QUICK_TEST_MODE else 300
MAX_HEALTH_TRAIN = 2000 if QUICK_TEST_MODE else 60000
MAX_HEALTH_TEST = 800 if QUICK_TEST_MODE else 25000
TOP_K = 5

PROFILES = ["general", "diabetes", "hypertension", "obesity", "child_nutrition", "senior_nutrition"]
NUTRIENTS = [
    "energy_kcal", "protein_g", "fiber_g", "potassium_mg", "calcium_mg", "iron_mg",
    "vitamin_c_mg", "sodium_mg", "saturated_fat_g", "total_sugars_g", "total_fat_g", "carbohydrate_g",
]
HEALTH_PROFILES_FOR_GROUPS = ["diabetes", "hypertension", "obesity", "child_nutrition", "senior_nutrition"]

# Embedded verbatim from the original project's config/health_scoring.yaml,
# so this notebook needs no yaml config file uploaded alongside the data.
NUTRIENT_CONFIG = {
    "protein_g": {"direction": "positive", "reference": 25.0},
    "fiber_g": {"direction": "positive", "reference": 10.0},
    "potassium_mg": {"direction": "positive", "reference": 1000.0},
    "calcium_mg": {"direction": "positive", "reference": 400.0},
    "iron_mg": {"direction": "positive", "reference": 8.0},
    "vitamin_c_mg": {"direction": "positive", "reference": 60.0},
    "sodium_mg": {"direction": "negative", "reference": 600.0},
    "saturated_fat_g": {"direction": "negative", "reference": 8.0},
    "total_sugars_g": {"direction": "negative", "reference": 20.0},
}
PROFILE_WEIGHTS = {
    "general": {"protein_g": 0.15, "fiber_g": 0.20, "potassium_mg": 0.12, "calcium_mg": 0.10,
                "iron_mg": 0.08, "vitamin_c_mg": 0.10, "sodium_mg": 0.10, "saturated_fat_g": 0.08,
                "total_sugars_g": 0.07},
    "diabetes": {"protein_g": 0.12, "fiber_g": 0.28, "potassium_mg": 0.10, "calcium_mg": 0.06,
                 "iron_mg": 0.04, "vitamin_c_mg": 0.08, "sodium_mg": 0.08, "saturated_fat_g": 0.08,
                 "total_sugars_g": 0.16},
    "hypertension": {"protein_g": 0.10, "fiber_g": 0.15, "potassium_mg": 0.22, "calcium_mg": 0.10,
                      "iron_mg": 0.04, "vitamin_c_mg": 0.08, "sodium_mg": 0.22, "saturated_fat_g": 0.06,
                      "total_sugars_g": 0.03},
    "obesity": {"protein_g": 0.18, "fiber_g": 0.24, "potassium_mg": 0.10, "calcium_mg": 0.06,
                "iron_mg": 0.04, "vitamin_c_mg": 0.08, "sodium_mg": 0.08, "saturated_fat_g": 0.10,
                "total_sugars_g": 0.12},
    "child_nutrition": {"protein_g": 0.20, "fiber_g": 0.14, "potassium_mg": 0.10, "calcium_mg": 0.18,
                         "iron_mg": 0.14, "vitamin_c_mg": 0.10, "sodium_mg": 0.06, "saturated_fat_g": 0.04,
                         "total_sugars_g": 0.04},
    "senior_nutrition": {"protein_g": 0.22, "fiber_g": 0.20, "potassium_mg": 0.15, "calcium_mg": 0.14,
                          "iron_mg": 0.05, "vitamin_c_mg": 0.08, "sodium_mg": 0.10, "saturated_fat_g": 0.04,
                          "total_sugars_g": 0.02},
}
RANKING_WEIGHTS = {"acceptance": 0.34, "health": 0.33, "similarity": 0.33}

RELATION_GROUPS = {
    "behavior": ["group_observed_food", "food_observed_group"],
    "nutrient": ["food_has_nutrient", "nutrient_in_food"],
    "category": ["food_in_category", "category_has_food"],
    "similarity": ["food_similar_to_food"],
}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [105]:
# ============================================================================
# 2. DATA DISCOVERY
# ============================================================================
def resolve_data_file(filename: str) -> Path:
    """Find a processed data file under /kaggle/input (any dataset name),
    or under MANUAL_DATA_DIR if set."""
    if MANUAL_DATA_DIR is not None:
        candidate = Path(MANUAL_DATA_DIR) / filename
        if candidate.exists():
            return candidate
    matches = sorted(glob.glob(f"/kaggle/input/**/{filename}", recursive=True))
    if matches:
        return Path(matches[0])
    matches = sorted(glob.glob(f"./**/{filename}", recursive=True))
    if matches:
        return Path(matches[0])
    raise FileNotFoundError(
        f"Could not find '{filename}'. Add your Kaggle Dataset containing it via "
        f"'+ Add Data', or set MANUAL_DATA_DIR at the top of the config cell."
    )


DB_PATH = resolve_data_file("food_relations.sqlite")
CATALOG_PATH = resolve_data_file("food_catalog.json")
PAIRS_PATH = resolve_data_file("substitution_pairs.json")
print("database:", DB_PATH)
print("catalog :", CATALOG_PATH)
print("pairs   :", PAIRS_PATH)

database: /kaggle/input/datasets/mdlimonbinhossain/foodrltndataset/food_relations.sqlite
catalog : /kaggle/input/datasets/mdlimonbinhossain/foodrltndataset/food_catalog.json
pairs   : /kaggle/input/datasets/mdlimonbinhossain/foodrltndataset/substitution_pairs.json


In [106]:
# ============================================================================
# 3. DATA LOADING  (ports load_prepared_data from the original project)
# ============================================================================
@dataclass
class PreparedData:
    table_counts: dict
    foods: pd.DataFrame
    groups: pd.DataFrame
    behavior: pd.DataFrame
    health_pivot: pd.DataFrame
    health_coverage: pd.DataFrame
    group_profile_mix: pd.DataFrame
    group_features: pd.DataFrame
    food_features: pd.DataFrame
    nutrient_features: pd.DataFrame
    category_features: pd.DataFrame
    food_nutrient_edges: pd.DataFrame
    food_category_edges: pd.DataFrame
    similarity: pd.DataFrame
    food_descriptions: dict


def read_sql(db_path: Path, query: str) -> pd.DataFrame:
    with sqlite3.connect(db_path) as con:
        return pd.read_sql_query(query, con)


def normalize_columns(values: pd.DataFrame) -> pd.DataFrame:
    arr = values.astype(float).replace([np.inf, -np.inf], np.nan)
    means = arr.mean(axis=0)
    arr = arr.fillna(means).fillna(0.0)
    stds = arr.std(axis=0).replace(0.0, 1.0).fillna(1.0)
    return (arr - means) / stds


def one_hot(frame: pd.DataFrame, columns: list) -> pd.DataFrame:
    return pd.get_dummies(frame[columns].astype(str), prefix=columns, dtype=float)


def load_prepared_data(db_path: Path, catalog_path: Path, pairs_path: Path) -> PreparedData:
    with sqlite3.connect(db_path) as con:
        table_counts = json.loads(dict(con.execute("select key, value from build_metadata").fetchall())["table_counts"])

    foods = read_sql(db_path, "select * from foods")
    groups = read_sql(db_path, "select * from household_groups")
    nutrients = read_sql(db_path, "select * from food_nutrients")
    health = read_sql(db_path, "select * from food_health_scores")
    behavior = read_sql(
        db_path,
        """
        select group_id, food_id,
               sum(occasions) as occasions,
               sum(total_grams) as total_grams,
               count(*) as source_rows
        from group_food_relations
        group by group_id, food_id
        """,
    )
    condition_counts = read_sql(
        db_path,
        """
        select hgm.group_id, pc.condition, count(distinct pc.person_id) as people
        from person_health_conditions pc
        join persons p on p.person_id = pc.person_id
        join household_group_memberships hgm on hgm.household_id = p.household_id
        group by hgm.group_id, pc.condition
        """,
    )
    group_people = read_sql(
        db_path,
        """
        select hgm.group_id, count(distinct p.person_id) as people
        from persons p
        join household_group_memberships hgm on hgm.household_id = p.household_id
        group by hgm.group_id
        """,
    )

    health_pivot = (
        health.pivot_table(index="food_id", columns="profile", values="health_score", aggfunc="mean")
        .reindex(columns=PROFILES)
        .dropna()
    )
    health_coverage = (
        health.pivot_table(index="food_id", columns="profile", values="weight_coverage", aggfunc="mean")
        .reindex(columns=PROFILES)
        .loc[health_pivot.index]
    )
    food_ids = set(health_pivot.index)
    foods = foods[foods["food_id"].isin(food_ids)].copy()
    behavior = behavior[
        behavior["food_id"].isin(food_ids) & behavior["group_id"].isin(set(groups["group_id"]))
    ].copy()
    behavior["behavior_strength"] = np.log1p(behavior["occasions"].clip(lower=0))
    max_by_group = behavior.groupby("group_id")["behavior_strength"].transform("max").replace(0, 1)
    behavior["behavior_strength"] = (behavior["behavior_strength"] / max_by_group).clip(0, 1)

    group_people_s = group_people.set_index("group_id")["people"].replace(0, np.nan)
    profile_mix = pd.DataFrame(0.0, index=groups["group_id"], columns=PROFILES)
    profile_mix["general"] = 1.0
    for row in condition_counts.itertuples(index=False):
        if row.condition in HEALTH_PROFILES_FOR_GROUPS and row.group_id in profile_mix.index:
            denom = float(group_people_s.get(row.group_id, np.nan))
            profile_mix.loc[row.group_id, row.condition] = 0.0 if math.isnan(denom) else row.people / denom
    profile_mix = profile_mix.div(profile_mix.sum(axis=1).replace(0, 1), axis=0)

    group_parts = groups["group_label"].str.split("|", expand=True)
    for idx, col in enumerate(["group_source", "poverty_band", "household_size_band", "snap_band"]):
        groups[col] = group_parts[idx] if idx in group_parts else "unknown"
    group_numeric = normalize_columns(
        groups.set_index("group_id")[
            ["member_count", "mean_household_size", "mean_income_poverty_ratio", "snap_share"]
        ]
    )
    group_feature_frame = pd.concat(
        [
            group_numeric,
            one_hot(groups.set_index("group_id"), ["group_source", "poverty_band", "household_size_band", "snap_band"]),
            profile_mix,
        ],
        axis=1,
    ).astype(float)

    nutrient_pivot = (
        nutrients[nutrients["nutrient"].isin(NUTRIENTS)]
        .pivot_table(index="food_id", columns="nutrient", values="amount_per_100g", aggfunc="mean")
        .reindex(index=health_pivot.index, columns=NUTRIENTS)
    )
    nutrient_log = np.log1p(nutrient_pivot.clip(lower=0))
    nutrient_missing = nutrient_pivot.isna().astype(float)
    nutrient_missing.columns = [f"missing_{col}" for col in nutrient_missing.columns]
    food_source = one_hot(foods.set_index("food_id"), ["source_release"])
    category_numeric = normalize_columns(foods.set_index("food_id")[["category_code"]])
    food_feature_frame = pd.concat(
        [normalize_columns(nutrient_log), nutrient_missing, category_numeric, food_source],
        axis=1,
    ).reindex(health_pivot.index).fillna(0.0)

    nutrient_features = []
    for nutrient in NUTRIENTS:
        details = NUTRIENT_CONFIG.get(nutrient, {})
        reference = float(details.get("reference", 1.0))
        direction = 1.0 if details.get("direction") == "positive" else -1.0
        mean_weight = np.mean([PROFILE_WEIGHTS.get(profile, {}).get(nutrient, 0.0) for profile in PROFILES])
        nutrient_features.append(
            {"nutrient": nutrient, "direction": direction, "log_reference": math.log1p(reference),
             "mean_profile_weight": mean_weight}
        )
    nutrient_feature_frame = pd.DataFrame(nutrient_features).set_index("nutrient")
    nutrient_feature_frame = normalize_columns(nutrient_feature_frame)

    category_counts = foods.groupby("category_code")["food_id"].count().rename("food_count")
    category_feature_frame = pd.DataFrame(index=category_counts.index.astype(str))
    category_feature_frame["category_code"] = pd.to_numeric(category_counts.index, errors="coerce")
    category_feature_frame["food_count"] = category_counts.to_numpy()
    category_feature_frame = normalize_columns(category_feature_frame)

    nutrient_edges = nutrients[nutrients["nutrient"].isin(NUTRIENTS) & nutrients["food_id"].isin(food_ids)].copy()
    references = {name: float(spec.get("reference", 1.0)) for name, spec in NUTRIENT_CONFIG.items()}
    nutrient_edges["edge_weight"] = [
        min(max(float(value), 0.0) / references.get(nutrient, 1.0), 1.0)
        for nutrient, value in zip(nutrient_edges["nutrient"], nutrient_edges["amount_per_100g"])
    ]
    nutrient_edges = nutrient_edges[nutrient_edges["edge_weight"] > 0].copy()

    category_edges = foods[["food_id", "category_code"]].dropna().copy()
    category_edges["category_id"] = category_edges["category_code"].astype(str)
    category_edges["edge_weight"] = 1.0

    catalog = json.loads(catalog_path.read_text(encoding="utf-8"))
    pairs = json.loads(pairs_path.read_text(encoding="utf-8"))
    fndds_2021 = {
        str(row.food_code): row.food_id
        for row in foods[foods["source_release"].eq("FNDDS_2021_2023")].itertuples(index=False)
    }
    fndds_any = {}
    for row in foods.itertuples(index=False):
        fndds_any.setdefault(str(row.food_code), row.food_id)
    catalog_to_db = {}
    for record in catalog["records"]:
        code = (record.get("nutrition_links") or {}).get("fndds_code")
        catalog_to_db[record["food_id"]] = None if code is None else fndds_2021.get(str(code), fndds_any.get(str(code)))

    similarity_rows = []
    for pair in pairs["pairs"]:
        request_food = catalog_to_db.get(pair["request_food_id"])
        substitute_food = catalog_to_db.get(pair["substitute_food_id"])
        if not request_food or not substitute_food or request_food == substitute_food:
            continue
        if request_food not in food_ids or substitute_food not in food_ids:
            continue
        similarity_rows.append(
            {
                "request_food_id": request_food,
                "substitute_food_id": substitute_food,
                "similarity_score": float(pair["similarity_score"]),
                "pair_id": pair["pair_id"],
            }
        )
    similarity = pd.DataFrame(similarity_rows).drop_duplicates(
        ["request_food_id", "substitute_food_id"], keep="first"
    )
    food_descriptions = dict(zip(foods["food_id"], foods["description"]))

    return PreparedData(
        table_counts=table_counts, foods=foods, groups=groups, behavior=behavior,
        health_pivot=health_pivot, health_coverage=health_coverage, group_profile_mix=profile_mix,
        group_features=group_feature_frame, food_features=food_feature_frame,
        nutrient_features=nutrient_feature_frame, category_features=category_feature_frame,
        food_nutrient_edges=nutrient_edges[["food_id", "nutrient", "edge_weight"]],
        food_category_edges=category_edges[["food_id", "category_id", "edge_weight"]],
        similarity=similarity, food_descriptions=food_descriptions,
    )


DATA = load_prepared_data(DB_PATH, CATALOG_PATH, PAIRS_PATH)
print(f"foods={len(DATA.food_features)}  groups={len(DATA.group_features)}  "
      f"behavior_edges={len(DATA.behavior)}  similarity_labels={len(DATA.similarity)}")

foods=8871  groups=42  behavior_edges=59512  similarity_labels=56


In [107]:
# ============================================================================
# 4. SPLITS  (standard gets a real train/val/test — the original code
#    discarded its middle 10% instead of using it for tuning; that gap is
#    now a proper validation set so Optuna never touches the test set)
# ============================================================================
@dataclass
class SplitSpec:
    name: str
    train_pos: pd.DataFrame
    eval_pos: pd.DataFrame
    train_groups: set
    eval_groups: set
    train_foods: set
    eval_foods: set
    graph_foods: set


def three_way_split(frame: pd.DataFrame, rng: np.random.Generator,
                     train_fraction: float = 0.70, val_fraction: float = 0.10):
    idx = np.arange(len(frame))
    rng.shuffle(idx)
    n = len(idx)
    train_end = int(n * train_fraction)
    val_end = train_end + int(n * val_fraction)
    return (
        frame.iloc[idx[:train_end]].copy(),
        frame.iloc[idx[train_end:val_end]].copy(),
        frame.iloc[idx[val_end:]].copy(),
    )


def two_way_split(frame: pd.DataFrame, rng: np.random.Generator, train_fraction: float = 0.70):
    idx = np.arange(len(frame))
    rng.shuffle(idx)
    train_end = int(len(idx) * train_fraction)
    return frame.iloc[idx[:train_end]].copy(), frame.iloc[idx[train_end:]].copy()


def make_all_splits(data: PreparedData, seed: int):
    """Returns (standard_hpo, standard_final, cold_food, cold_group) SplitSpecs.
    standard_hpo   : train -> val   (used only for Optuna search + pruning)
    standard_final : train+val -> test (used for the reported final model)
    """
    rng = np.random.default_rng(seed)
    all_groups = set(data.group_features.index)
    all_foods = set(data.food_features.index)

    std_train, std_val, std_test = three_way_split(data.behavior, rng)

    standard_hpo = SplitSpec(
        name="standard_hpo", train_pos=std_train, eval_pos=std_val,
        train_groups=all_groups, eval_groups=all_groups,
        train_foods=all_foods, eval_foods=all_foods, graph_foods=all_foods,
    )
    standard_final = SplitSpec(
        name="standard", train_pos=pd.concat([std_train, std_val], ignore_index=True), eval_pos=std_test,
        train_groups=all_groups, eval_groups=all_groups,
        train_foods=all_foods, eval_foods=all_foods, graph_foods=all_foods,
    )

    # cold-start-food: entire food nodes withheld from behavior supervision.
    sim_foods = set(data.similarity["request_food_id"]).union(set(data.similarity["substitute_food_id"]))
    other_foods = sorted(all_foods - sim_foods)
    sim_food_list = sorted(sim_foods)
    rng.shuffle(sim_food_list)
    rng.shuffle(other_foods)
    cold_sim_count = max(1, int(round(0.20 * len(sim_food_list)))) if sim_food_list else 0
    cold_other_count = int(round(0.20 * len(other_foods)))
    cold_foods = set(sim_food_list[:cold_sim_count]).union(other_foods[:cold_other_count])
    train_foods = all_foods - cold_foods
    cold_food = SplitSpec(
        name="cold_start_food",
        train_pos=data.behavior[data.behavior["food_id"].isin(train_foods)].copy(),
        eval_pos=data.behavior[data.behavior["food_id"].isin(cold_foods)].copy(),
        train_groups=all_groups, eval_groups=all_groups,
        train_foods=train_foods, eval_foods=cold_foods, graph_foods=train_foods,
    )

    # cold-start-household-group: entire group nodes withheld from behavior supervision.
    group_list = sorted(all_groups)
    rng.shuffle(group_list)
    cold_group_count = max(1, int(round(0.20 * len(group_list))))
    cold_groups = set(group_list[:cold_group_count])
    train_groups = all_groups - cold_groups
    cold_group = SplitSpec(
        name="cold_start_group",
        train_pos=data.behavior[data.behavior["group_id"].isin(train_groups)].copy(),
        eval_pos=data.behavior[data.behavior["group_id"].isin(cold_groups)].copy(),
        train_groups=train_groups, eval_groups=cold_groups,
        train_foods=all_foods, eval_foods=all_foods, graph_foods=all_foods,
    )
    return standard_hpo, standard_final, cold_food, cold_group

In [108]:
# ============================================================================
# 5. EXAMPLE / NEGATIVE SAMPLING BUILDERS  (unchanged logic, ported)
# ============================================================================
def sample_behavior_negatives(observed: set, groups: list, foods: list, n_samples: int, rng: np.random.Generator) -> pd.DataFrame:
    rows, seen = [], set()
    while len(rows) < n_samples:
        batch = max(2000, (n_samples - len(rows)) * 3)
        group_draws = rng.choice(groups, size=batch, replace=True)
        food_draws = rng.choice(foods, size=batch, replace=True)
        for group_id, food_id in zip(group_draws, food_draws):
            key = (str(group_id), str(food_id))
            if key in observed or key in seen:
                continue
            seen.add(key)
            rows.append(key)
            if len(rows) >= n_samples:
                break
    return pd.DataFrame(rows, columns=["group_id", "food_id"])


def behavior_examples(positive_edges, observed, groups, foods, negative_ratio, rng):
    pos = positive_edges[positive_edges["group_id"].isin(groups) & positive_edges["food_id"].isin(foods)].copy()
    pos = pos[["group_id", "food_id", "behavior_strength"]].rename(columns={"behavior_strength": "target"})
    pos["binary_target"] = 1.0
    neg = sample_behavior_negatives(observed, sorted(groups), sorted(foods), len(pos) * negative_ratio, rng)
    neg["target"] = 0.0
    neg["binary_target"] = 0.0
    return pd.concat([pos, neg], ignore_index=True)


def sample_health_examples(group_ids, food_ids, health_targets, max_examples, rng):
    groups = sorted(group_ids)
    foods = sorted(food_ids)
    total = len(groups) * len(foods)
    n = min(max_examples, total)
    flat_indices = rng.choice(total, size=n, replace=False)
    group_idx = flat_indices // len(foods)
    food_idx = flat_indices % len(foods)
    rows = []
    for gi, fi in zip(group_idx, food_idx):
        group_id = groups[int(gi)]
        food_id = foods[int(fi)]
        rows.append((group_id, food_id, float(health_targets.loc[group_id, food_id])))
    return pd.DataFrame(rows, columns=["group_id", "food_id", "target"])


def split_similarity_examples(positives, train_foods, test_foods, split_name, all_foods, rng, negative_ratio=4):
    pos = positives.copy()
    if split_name == "cold_start_food":
        train_pos = pos[pos["request_food_id"].isin(train_foods) & pos["substitute_food_id"].isin(train_foods)].copy()
        test_pos = pos[pos["request_food_id"].isin(test_foods) | pos["substitute_food_id"].isin(test_foods)].copy()
        if len(test_pos) == 0:
            train_pos, test_pos = two_way_split(pos, rng)
    else:
        train_pos, test_pos = two_way_split(pos, rng)

    observed = set(zip(pos["request_food_id"], pos["substitute_food_id"]))

    def with_negatives(source, food_pool):
        positive_examples = source[["request_food_id", "substitute_food_id", "similarity_score"]].rename(
            columns={"similarity_score": "target"}
        )
        count = len(positive_examples) * negative_ratio
        if count == 0:
            return positive_examples
        foods = sorted(food_pool)
        negatives, seen = [], set()
        while len(negatives) < count and len(foods) > 1:
            a = str(rng.choice(foods))
            b = str(rng.choice(foods))
            key = (a, b)
            if a == b or key in observed or key in seen:
                continue
            seen.add(key)
            negatives.append((a, b, 0.0))
        neg_frame = pd.DataFrame(negatives, columns=["request_food_id", "substitute_food_id", "target"])
        return pd.concat([positive_examples, neg_frame], ignore_index=True).drop_duplicates(
            ["request_food_id", "substitute_food_id"], keep="first"
        )

    train_examples = with_negatives(train_pos, train_foods)
    test_pool = test_foods if split_name == "cold_start_food" else all_foods
    test_examples = with_negatives(test_pos, test_pool)
    return train_examples, test_examples


def make_index_tensor(frame, left_col, right_col, left_index, right_index):
    pairs = [(left_index[row[left_col]], right_index[row[right_col]]) for row in frame.to_dict("records")]
    return torch.tensor(pairs, dtype=torch.long)

In [109]:
# ============================================================================
# 6. GRAPH CONSTRUCTION  (adds disabled_relations= for the ablation study)
# ============================================================================
@dataclass
class GraphData:
    node_features: dict
    edges: dict
    node_ids: dict


def make_graph(data: PreparedData, split: SplitSpec, disabled_relations: Optional[set] = None) -> tuple:
    disabled_relations = disabled_relations or set()
    group_ids = list(data.group_features.index)
    food_ids = list(data.food_features.index)
    nutrient_ids = list(data.nutrient_features.index)
    category_ids = list(data.category_features.index)
    indices = {
        "household": {item: idx for idx, item in enumerate(group_ids)},
        "food": {item: idx for idx, item in enumerate(food_ids)},
        "nutrient": {item: idx for idx, item in enumerate(nutrient_ids)},
        "category": {item: idx for idx, item in enumerate(category_ids)},
    }
    node_features = {
        "household": torch.tensor(data.group_features.loc[group_ids].to_numpy(dtype=np.float32)),
        "food": torch.tensor(data.food_features.loc[food_ids].to_numpy(dtype=np.float32)),
        "nutrient": torch.tensor(data.nutrient_features.loc[nutrient_ids].to_numpy(dtype=np.float32)),
        "category": torch.tensor(data.category_features.loc[category_ids].to_numpy(dtype=np.float32)),
    }

    edges = {}

    def add_edges(relation, source_type, target_type, rows):
        if not rows:
            edge_index = torch.empty((2, 0), dtype=torch.long)
            edge_weight = torch.empty((0,), dtype=torch.float32)
        else:
            edge_index = torch.tensor(
                [[indices[source_type][s] for s, _, _ in rows], [indices[target_type][t] for _, t, _ in rows]],
                dtype=torch.long,
            )
            edge_weight = torch.tensor([w for _, _, w in rows], dtype=torch.float32)
        edges[relation] = (source_type, target_type, edge_index, edge_weight)

    graph_foods = split.graph_foods

    if "behavior" not in disabled_relations:
        behavior_rows = [
            (row.group_id, row.food_id, float(row.behavior_strength))
            for row in split.train_pos.itertuples(index=False)
            if row.group_id in split.train_groups and row.food_id in graph_foods
        ]
        add_edges("group_observed_food", "household", "food", behavior_rows)
        add_edges("food_observed_group", "food", "household", [(b, a, w) for a, b, w in behavior_rows])

    if "nutrient" not in disabled_relations:
        nutrient_rows = [
            (row.food_id, row.nutrient, float(row.edge_weight))
            for row in data.food_nutrient_edges.itertuples(index=False)
            if row.food_id in graph_foods
        ]
        add_edges("food_has_nutrient", "food", "nutrient", nutrient_rows)
        add_edges("nutrient_in_food", "nutrient", "food", [(b, a, w) for a, b, w in nutrient_rows])

    if "category" not in disabled_relations:
        category_rows = [
            (row.food_id, row.category_id, float(row.edge_weight))
            for row in data.food_category_edges.itertuples(index=False)
            if row.food_id in graph_foods and row.category_id in indices["category"]
        ]
        add_edges("food_in_category", "food", "category", category_rows)
        add_edges("category_has_food", "category", "food", [(b, a, w) for a, b, w in category_rows])

    if "similarity" not in disabled_relations:
        sim_rows = [
            (row.request_food_id, row.substitute_food_id, float(row.similarity_score))
            for row in data.similarity.itertuples(index=False)
            if row.request_food_id in graph_foods and row.substitute_food_id in graph_foods
        ]
        add_edges("food_similar_to_food", "food", "food", sim_rows)

    graph = GraphData(
        node_features=node_features, edges=edges,
        node_ids={"household": group_ids, "food": food_ids, "nutrient": nutrient_ids, "category": category_ids},
    )
    return graph, indices

In [110]:
# ============================================================================
# 7. MODEL — upgraded heterogeneous GNN
#    New vs. the original graph_recommender.py: residual connections,
#    selectable activation (relu/gelu), selectable message aggregation
#    (mean/sum), and a searchable head width ratio. All four are exposed
#    to Optuna in Section 9 as genuine architecture search dimensions,
#    not just width/depth/dropout.
# ============================================================================
ACTIVATIONS = {"relu": torch.relu, "gelu": F.gelu}


class HeterogeneousLayer(nn.Module):
    def __init__(self, hidden_dim: int, relations, activation: str = "relu", aggregation: str = "mean"):
        super().__init__()
        self.self_linears = nn.ModuleDict()
        self.relation_linears = nn.ModuleDict(
            {relation: nn.Linear(hidden_dim, hidden_dim, bias=False) for relation in relations}
        )
        self.norms = nn.ModuleDict()
        self.activation = ACTIVATIONS[activation]
        self.aggregation = aggregation
        self._hidden_dim = hidden_dim

    def initialize_node_types(self, node_types) -> None:
        for node_type in node_types:
            self.self_linears[node_type] = nn.Linear(self._hidden_dim, self._hidden_dim)
            self.norms[node_type] = nn.LayerNorm(self._hidden_dim)

    def forward(self, states: dict, edges: dict) -> dict:
        outputs = {nt: self.self_linears[nt](v) for nt, v in states.items()}
        counts = {nt: torch.ones((v.shape[0], 1), dtype=v.dtype, device=v.device) for nt, v in states.items()}
        for relation, (source_type, target_type, edge_index, edge_weight) in edges.items():
            if edge_index.numel() == 0:
                continue
            source_index, target_index = edge_index
            messages = self.relation_linears[relation](states[source_type][source_index])
            messages = messages * edge_weight.unsqueeze(1)
            outputs[target_type].index_add_(0, target_index, messages)
            if self.aggregation == "mean":
                counts[target_type].index_add_(0, target_index, edge_weight.abs().unsqueeze(1))
        if self.aggregation == "mean":
            pooled = {nt: v / counts[nt].clamp_min(1.0) for nt, v in outputs.items()}
        else:  # "sum": no normalization by neighbor count
            pooled = outputs
        return {nt: self.activation(self.norms[nt](v)) for nt, v in pooled.items()}


class MultiObjectiveHeteroGNN(nn.Module):
    def __init__(self, input_dims: dict, relations, hidden_dim: int = 64, layers: int = 2,
                 dropout: float = 0.1, residual: bool = False, activation: str = "relu",
                 aggregation: str = "mean", head_hidden_ratio: int = 2):
        super().__init__()
        relation_names = list(relations)
        self.encoders = nn.ModuleDict({nt: nn.Linear(size, hidden_dim) for nt, size in input_dims.items()})
        self.layers = nn.ModuleList()
        for _ in range(layers):
            layer = HeterogeneousLayer(hidden_dim, relation_names, activation=activation, aggregation=aggregation)
            layer.initialize_node_types(input_dims)
            self.layers.append(layer)
        self.dropout = nn.Dropout(dropout)
        self.residual = residual
        pair_dim = hidden_dim * 2
        self.acceptance_head = self._head(pair_dim, head_hidden_ratio)
        self.health_head = self._head(pair_dim, head_hidden_ratio)
        self.similarity_head = self._head(pair_dim, head_hidden_ratio)

    @staticmethod
    def _head(input_dim: int, ratio: int) -> nn.Sequential:
        hidden = max(4, input_dim // ratio)
        return nn.Sequential(nn.Linear(input_dim, hidden), nn.ReLU(), nn.Linear(hidden, 1))

    def encode(self, graph: GraphData) -> dict:
        states = {nt: torch.relu(self.encoders[nt](feats)) for nt, feats in graph.node_features.items()}
        for layer in self.layers:
            new_states = layer(states, graph.edges)
            if self.residual:
                new_states = {nt: new_states[nt] + states[nt] for nt in new_states}
            states = {nt: self.dropout(v) for nt, v in new_states.items()}
        return states

    @staticmethod
    def _pairs(states, left_type, right_type, indices):
        return torch.cat([states[left_type][indices[:, 0]], states[right_type][indices[:, 1]]], dim=1)

    def predict_acceptance(self, states, indices):
        return torch.sigmoid(self.acceptance_head(self._pairs(states, "household", "food", indices))).squeeze(1)

    def predict_health(self, states, indices):
        return torch.sigmoid(self.health_head(self._pairs(states, "household", "food", indices))).squeeze(1)

    def predict_similarity(self, states, indices):
        return torch.sigmoid(self.similarity_head(self._pairs(states, "food", "food", indices))).squeeze(1)

In [111]:
# ============================================================================
# 8. METRICS + TRAIN/EVAL CORE  (shared by Optuna trials, final training,
#    and the relation-ablation study)
# ============================================================================
def binary_metrics(frame: pd.DataFrame, scores: np.ndarray) -> dict:
    y = frame["binary_target"].to_numpy(dtype=float)
    clipped = np.clip(scores, 1e-6, 1 - 1e-6)
    if len(np.unique(y)) < 2:
        return {"roc_auc": float("nan"), "pr_auc": float("nan"), "log_loss": float("nan"),
                "brier": float("nan")}
    return {
        "roc_auc": float(roc_auc_score(y, scores)),
        "pr_auc": float(average_precision_score(y, scores)),
        "log_loss": float(log_loss(y, clipped)),
        "brier": float(brier_score_loss(y, scores)),
    }


def behavior_ranking_metrics(frame: pd.DataFrame, scores: np.ndarray, top_k: int) -> dict:
    values = frame.copy()
    values["score"] = scores
    precisions, recalls, ndcgs = [], [], []
    for _, group in values.groupby("group_id"):
        positives = float(group["binary_target"].sum())
        if positives <= 0:
            continue
        ranked = group.sort_values("score", ascending=False)
        top = ranked.head(top_k)
        precisions.append(float(top["binary_target"].sum()) / top_k)
        recalls.append(float(top["binary_target"].sum()) / positives)
        gains = top["binary_target"].to_numpy(dtype=float)
        discounts = 1.0 / np.log2(np.arange(2, len(gains) + 2))
        dcg = float(np.sum(gains * discounts))
        ideal_n = min(int(positives), top_k)
        ideal = float(np.sum(discounts[:ideal_n]))
        ndcgs.append(dcg / ideal if ideal > 0 else 0.0)
    return {
        f"precision_at_{top_k}": float(np.mean(precisions)) if precisions else float("nan"),
        f"recall_at_{top_k}": float(np.mean(recalls)) if recalls else float("nan"),
        f"ndcg_at_{top_k}": float(np.mean(ndcgs)) if ndcgs else float("nan"),
    }


def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    return {
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(math.sqrt(mean_squared_error(y_true, y_pred))),
        "spearman": float(pd.Series(y_true).corr(pd.Series(y_pred), method="spearman")),
    }


@dataclass
class RunResult:
    metrics: dict
    history: pd.DataFrame
    model: MultiObjectiveHeteroGNN
    graph: GraphData
    indices: dict
    test_behavior: pd.DataFrame = None
    behavior_scores: np.ndarray = None
    test_health: pd.DataFrame = None
    health_scores: np.ndarray = None
    test_sim: pd.DataFrame = None
    sim_scores: np.ndarray = None


def train_and_evaluate(
    data: PreparedData, split: SplitSpec, hp: dict, seed: int, epochs: int,
    negative_ratio: int, max_health_train: int, max_health_test: int, top_k: int,
    disabled_relations: Optional[set] = None, trial: Optional["optuna.Trial"] = None,
    prune_every: int = 5, verbose: bool = False,
) -> RunResult:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)

    graph, indices = make_graph(data, split, disabled_relations=disabled_relations)
    for nt in graph.node_features:
        graph.node_features[nt] = graph.node_features[nt].to(DEVICE)
    for rel in graph.edges:
        s, t, ei, ew = graph.edges[rel]
        graph.edges[rel] = (s, t, ei.to(DEVICE), ew.to(DEVICE))

    observed = set(zip(data.behavior["group_id"], data.behavior["food_id"]))
    train_behavior = behavior_examples(split.train_pos, observed, split.train_groups, split.train_foods, negative_ratio, rng)
    test_behavior = behavior_examples(split.eval_pos, observed, split.eval_groups, split.eval_foods, negative_ratio, rng)

    health_targets = data.group_profile_mix[PROFILES].to_numpy() @ data.health_pivot[PROFILES].T.to_numpy()
    health_targets = pd.DataFrame(health_targets, index=data.group_profile_mix.index, columns=data.health_pivot.index)
    train_health = sample_health_examples(split.train_groups, split.train_foods, health_targets, max_health_train, rng)
    test_health = sample_health_examples(split.eval_groups, split.eval_foods, health_targets, max_health_test, rng)

    train_sim, test_sim = split_similarity_examples(
        data.similarity, train_foods=split.train_foods, test_foods=split.eval_foods,
        split_name=split.name, all_foods=set(data.food_features.index), rng=rng, negative_ratio=4,
    )

    b_train_idx = make_index_tensor(train_behavior, "group_id", "food_id", indices["household"], indices["food"]).to(DEVICE)
    h_train_idx = make_index_tensor(train_health, "group_id", "food_id", indices["household"], indices["food"]).to(DEVICE)
    s_train_idx = make_index_tensor(train_sim, "request_food_id", "substitute_food_id", indices["food"], indices["food"]).to(DEVICE)
    b_test_idx = make_index_tensor(test_behavior, "group_id", "food_id", indices["household"], indices["food"]).to(DEVICE)
    h_test_idx = make_index_tensor(test_health, "group_id", "food_id", indices["household"], indices["food"]).to(DEVICE)
    s_test_idx = make_index_tensor(test_sim, "request_food_id", "substitute_food_id", indices["food"], indices["food"]).to(DEVICE)
    b_y = torch.tensor(train_behavior["binary_target"].to_numpy(dtype=np.float32)).to(DEVICE)
    h_y = torch.tensor(train_health["target"].to_numpy(dtype=np.float32)).to(DEVICE)
    s_y = torch.tensor(train_sim["target"].to_numpy(dtype=np.float32)).to(DEVICE)

    model = MultiObjectiveHeteroGNN(
        input_dims={nt: t.shape[1] for nt, t in graph.node_features.items()},
        relations=graph.edges.keys(),
        hidden_dim=hp["hidden_dim"], layers=hp["layers"], dropout=hp["dropout"],
        residual=hp["residual"], activation=hp["activation"], aggregation=hp["aggregation"],
        head_hidden_ratio=hp["head_hidden_ratio"],
    ).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=hp["lr"], weight_decay=hp["weight_decay"])

    history_rows = []
    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        states = model.encode(graph)
        behavior_pred = model.predict_acceptance(states, b_train_idx)
        health_pred = model.predict_health(states, h_train_idx)
        similarity_pred = model.predict_similarity(states, s_train_idx)
        loss_behavior = F.binary_cross_entropy(behavior_pred, b_y)
        loss_health = F.mse_loss(health_pred, h_y)
        loss_similarity = F.mse_loss(similarity_pred, s_y)
        loss = hp["w_acceptance"] * loss_behavior + hp["w_health"] * loss_health + hp["w_similarity"] * loss_similarity
        loss.backward()
        optimizer.step()

        if epoch == 1 or epoch % 5 == 0 or epoch == epochs:
            history_rows.append({
                "split": split.name, "epoch": epoch, "total_loss": float(loss.detach()),
                "behavior_loss": float(loss_behavior.detach()), "health_loss": float(loss_health.detach()),
                "similarity_loss": float(loss_similarity.detach()),
            })
            if verbose:
                print(f"  [{split.name}] epoch {epoch}/{epochs}  loss={float(loss.detach()):.4f}")

        if trial is not None and (epoch % prune_every == 0 or epoch == epochs):
            model.eval()
            with torch.no_grad():
                states = model.encode(graph)
                val_behavior_scores = model.predict_acceptance(states, b_test_idx).cpu().numpy()
                val_health_scores = model.predict_health(states, h_test_idx).cpu().numpy()
            bm = binary_metrics(test_behavior, val_behavior_scores)
            hm = regression_metrics(test_health["target"].to_numpy(dtype=float), val_health_scores)
            auc = bm["roc_auc"] if not math.isnan(bm["roc_auc"]) else 0.5
            composite = auc + (1.0 - min(hm["mae"], 1.0))
            trial.report(composite, epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()

    model.eval()
    with torch.no_grad():
        states = model.encode(graph)
        behavior_scores = model.predict_acceptance(states, b_test_idx).cpu().numpy()
        health_scores = model.predict_health(states, h_test_idx).cpu().numpy()
        similarity_scores = model.predict_similarity(states, s_test_idx).cpu().numpy()

    metrics = {"split": split.name}
    for key, value in binary_metrics(test_behavior, behavior_scores).items():
        metrics[f"behavior_{key}"] = value
    for key, value in behavior_ranking_metrics(test_behavior, behavior_scores, top_k).items():
        metrics[f"behavior_{key}"] = value
    for key, value in regression_metrics(test_health["target"].to_numpy(dtype=float), health_scores).items():
        metrics[f"health_{key}"] = value
    if len(test_sim) > 0 and test_sim["target"].nunique() > 1:
        for key, value in regression_metrics(test_sim["target"].to_numpy(dtype=float), similarity_scores).items():
            metrics[f"similarity_{key}"] = value
    else:
        for key in ["mae", "rmse", "spearman"]:
            metrics[f"similarity_{key}"] = float("nan")
    metrics.update({
        "train_behavior_positive_edges": int(len(split.train_pos)), "eval_behavior_positive_edges": int(len(split.eval_pos)),
        "train_health_examples": int(len(train_health)), "eval_health_examples": int(len(test_health)),
        "train_similarity_examples": int(len(train_sim)), "eval_similarity_examples": int(len(test_sim)),
        "train_groups": int(len(split.train_groups)), "eval_groups": int(len(split.eval_groups)),
        "train_foods": int(len(split.train_foods)), "eval_foods": int(len(split.eval_foods)),
    })

    return RunResult(
        metrics=metrics, history=pd.DataFrame(history_rows), model=model, graph=graph, indices=indices,
        test_behavior=test_behavior, behavior_scores=behavior_scores,
        test_health=test_health, health_scores=health_scores,
        test_sim=test_sim, sim_scores=similarity_scores,
    )

In [123]:
# ============================================================================
# 9. OPTUNA — BAYESIAN ARCHITECTURE + HYPERPARAMETER SEARCH
#    TPE sampler (Bayesian) + median pruning. Objective = behavior ROC AUC
#    + (1 - health MAE), evaluated on the VALIDATION split only (the test
#    split is never touched here). Similarity is left out of the HPO
#    objective on purpose: only 56 labels exist project-wide, so carving
#    out yet another slice for tuning would leave too little signal in
#    either half — see the paper's own "Limitations" section.
# ============================================================================
def build_hp(trial: "optuna.Trial") -> dict:
    return dict(
        hidden_dim=trial.suggest_categorical("hidden_dim", [32, 64, 96, 128]),
        layers=trial.suggest_int("layers", 1, 3),
        dropout=trial.suggest_float("dropout", 0.0, 0.5),
        residual=trial.suggest_categorical("residual", [True, False]),
        activation=trial.suggest_categorical("activation", ["relu", "gelu"]),
        aggregation=trial.suggest_categorical("aggregation", ["mean", "sum"]),
        head_hidden_ratio=trial.suggest_categorical("head_hidden_ratio", [2, 4]),
        lr=trial.suggest_float("lr", 1e-4, 5e-2, log=True),
        weight_decay=trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True),
        negative_ratio=trial.suggest_int("negative_ratio", 1, 4),
        w_acceptance=trial.suggest_float("w_acceptance", 0.5, 2.0),
        w_health=trial.suggest_float("w_health", 0.5, 2.0),
        w_similarity=trial.suggest_float("w_similarity", 0.5, 2.0),
    )


def objective(trial: "optuna.Trial") -> float:
    hp = build_hp(trial)
    result = train_and_evaluate(
        DATA, STANDARD_HPO, hp, seed=SEED + trial.number, epochs=HPO_EPOCHS,
        negative_ratio=hp["negative_ratio"], max_health_train=MAX_HEALTH_TRAIN // 4,
        max_health_test=MAX_HEALTH_TEST // 4, top_k=TOP_K, trial=trial, prune_every=5,
    )
    auc = result.metrics["behavior_roc_auc"]
    auc = 0.5 if math.isnan(auc) else auc
    mae = result.metrics["health_mae"]
    return auc + (1.0 - min(mae, 1.0))


STANDARD_HPO, STANDARD_FINAL, COLD_FOOD, COLD_GROUP = make_all_splits(DATA, SEED)

study = optuna.create_study(
    direction="maximize",
    sampler=TPESampler(seed=SEED),
    pruner=MedianPruner(n_warmup_steps=10),
    study_name="food_substitution_gnn_architecture_search",
)
study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=False)

print(f"Best trial: value={study.best_value:.4f}")
print(f"Best hyperparameters: {json.dumps(study.best_params, indent=2)}")

trials_df = study.trials_dataframe()
trials_df.to_csv(OUTPUT_DIR / "tables" / "optuna_trials.csv", index=False)

BEST_HP = dict(
    hidden_dim=study.best_params["hidden_dim"], layers=study.best_params["layers"],
    dropout=study.best_params["dropout"], residual=study.best_params["residual"],
    activation=study.best_params["activation"], aggregation=study.best_params["aggregation"],
    head_hidden_ratio=study.best_params["head_hidden_ratio"], lr=study.best_params["lr"],
    weight_decay=study.best_params["weight_decay"], w_acceptance=study.best_params["w_acceptance"],
    w_health=study.best_params["w_health"], w_similarity=study.best_params["w_similarity"],
)
BEST_NEGATIVE_RATIO = study.best_params["negative_ratio"]
(OUTPUT_DIR / "tables" / "best_hyperparameters.json").write_text(
    json.dumps({**BEST_HP, "negative_ratio": BEST_NEGATIVE_RATIO}, indent=2), encoding="utf-8"
)
import matplotlib as mpl

mpl.rcdefaults()

plt.style.use("default")

fig, ax = plt.subplots(figsize=(7,4.5))

fig.patch.set_facecolor("white")
ax.set_facecolor("white")
ax.grid(False)
# Optuna diagnostic plots
try:
    from optuna.visualization.matplotlib import plot_optimization_history, plot_param_importances
    fig = plot_optimization_history(study).figure
    fig.savefig(OUTPUT_DIR / "figures" / "optuna_optimization_history.png", dpi=600, bbox_inches="tight")
    plt.close(fig)
    fig = plot_param_importances(study).figure
    fig.savefig(OUTPUT_DIR / "figures" / "optuna_param_importances.png", dpi=600, bbox_inches="tight")
    plt.close(fig)
except (ImportError, ValueError, RuntimeError) as exc:
    print(f"(skipped optuna visualization plots: {exc})")

[I 2026-08-03 00:49:04,601] A new study created in memory with name: food_substitution_gnn_architecture_search
[I 2026-08-03 00:49:09,974] Trial 0 finished with value: 1.6483692493120548 and parameters: {'hidden_dim': 96, 'layers': 2, 'dropout': 0.49377524711883497, 'residual': False, 'activation': 'gelu', 'aggregation': 'sum', 'head_hidden_ratio': 4, 'lr': 0.0001349034813061554, 'weight_decay': 6.029351054197822e-05, 'negative_ratio': 3, 'w_acceptance': 1.045571945398493, 'w_health': 1.5417396199239315, 'w_similarity': 1.885052123065428}. Best is trial 0 with value: 1.6483692493120548.
[I 2026-08-03 00:49:14,991] Trial 1 finished with value: 1.7311207063818683 and parameters: {'hidden_dim': 128, 'layers': 1, 'dropout': 0.483432884760743, 'residual': True, 'activation': 'gelu', 'aggregation': 'mean', 'head_hidden_ratio': 2, 'lr': 0.0004951268735691401, 'weight_decay': 0.00010767456422886725, 'negative_ratio': 3, 'w_acceptance': 1.2352128285221586, 'w_health': 0.8107700003221329, 'w_sim

Best trial: value=1.8716
Best hyperparameters: {
  "hidden_dim": 64,
  "layers": 2,
  "dropout": 0.21768432891642056,
  "residual": true,
  "activation": "relu",
  "aggregation": "sum",
  "head_hidden_ratio": 2,
  "lr": 0.003480961488805864,
  "weight_decay": 1.338493154542602e-06,
  "negative_ratio": 2,
  "w_acceptance": 1.9055752192176885,
  "w_health": 1.3353701724932696,
  "w_similarity": 1.0828257706200823
}


/tmp/ipykernel_58/1581317928.py:86: ExperimentalWarning: optuna.visualization.matplotlib._param_importances.plot_param_importances is experimental (supported from v2.2.0). The interface can change in the future.
  fig = plot_param_importances(study).figure


In [113]:
# ============================================================================
# 10. FINAL TRAINING — best hyperparameters, full epoch budget, all three
#    splits (standard / cold-start-food / cold-start-group), matching the
#    original paper's evaluation protocol. Checkpoints saved for each.
# ============================================================================
FINAL_SPLITS = {"standard": STANDARD_FINAL, "cold_start_food": COLD_FOOD, "cold_start_group": COLD_GROUP}
FINAL_RESULTS: dict = {}

for i, (name, split) in enumerate(FINAL_SPLITS.items()):
    print(f"training final model: {name} ...")
    result = train_and_evaluate(
        DATA, split, BEST_HP, seed=SEED + i * 17, epochs=FINAL_EPOCHS,
        negative_ratio=BEST_NEGATIVE_RATIO, max_health_train=MAX_HEALTH_TRAIN,
        max_health_test=MAX_HEALTH_TEST, top_k=TOP_K, verbose=False,
    )
    FINAL_RESULTS[name] = result
    torch.save(
        {
            "model_state_dict": result.model.state_dict(),
            "hyperparameters": BEST_HP,
            "negative_ratio": BEST_NEGATIVE_RATIO,
            "split": name,
            "node_ids": result.graph.node_ids,
            "input_dims": {nt: t.shape[1] for nt, t in result.graph.node_features.items()},
            "relations": list(result.graph.edges.keys()),
            "metrics": result.metrics,
        },
        OUTPUT_DIR / "checkpoints" / f"gnn_{name}.pt",
    )

metrics_frame = pd.DataFrame([FINAL_RESULTS[name].metrics for name in FINAL_SPLITS])
metrics_frame.to_csv(OUTPUT_DIR / "tables" / "final_validation_results.csv", index=False)
history_frame = pd.concat([FINAL_RESULTS[name].history for name in FINAL_SPLITS], ignore_index=True)
history_frame.to_csv(OUTPUT_DIR / "tables" / "final_training_history.csv", index=False)
print(metrics_frame[["split", "behavior_roc_auc", "behavior_ndcg_at_5", "health_mae", "health_spearman",
                      "similarity_mae", "similarity_spearman"]].to_string(index=False))

STANDARD_RESULT = FINAL_RESULTS["standard"]

fig, ax = plt.subplots(figsize=(7, 4.5))
for name in FINAL_SPLITS:
    h = FINAL_RESULTS[name].history
    ax.plot(h["epoch"], h["total_loss"], label=name.replace("_", " ").title(), marker="o", markersize=3)
ax.set_xlabel("Epoch")
ax.set_ylabel("Joint training loss")
ax.set_title("Joint Training Loss by Split (best Optuna architecture)")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "figures" / "01_training_loss.png", dpi=600)
plt.close(fig)

fig, ax = plt.subplots(figsize=(7, 4.5))
plot_metrics = metrics_frame.set_index("split")[["behavior_roc_auc", "behavior_ndcg_at_5", "health_spearman"]]
plot_metrics.plot(kind="bar", ax=ax)
ax.set_ylabel("Score")
ax.set_title("Behavior and Health Validation Metrics by Split")
ax.set_xticklabels([s.replace("_", " ").title() for s in plot_metrics.index], rotation=0)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "figures" / "02_validation_metrics.png", dpi=600)
plt.close(fig)

training final model: standard ...
training final model: cold_start_food ...
training final model: cold_start_group ...
           split  behavior_roc_auc  behavior_ndcg_at_5  health_mae  health_spearman  similarity_mae  similarity_spearman
        standard          0.947736            0.996876    0.029001         0.939289        0.037708             0.693842
 cold_start_food          0.749144            0.483525    0.071144         0.898379        0.126249             0.665398
cold_start_group          0.471291            0.250000    0.025030         0.939715        0.039479             0.692618


In [114]:
# ============================================================================
# 11. CANDIDATE SCORING  (predict all three objectives for every mapped
#    request/substitute pair, using the final standard-split model — this
#    replaces compute_recommendation_summary from the original script)
# ============================================================================
def score_all_candidates(data: PreparedData, result: RunResult) -> pd.DataFrame:
    model, graph, indices = result.model, result.graph, result.indices
    health_targets = pd.DataFrame(
        data.group_profile_mix[PROFILES].to_numpy() @ data.health_pivot[PROFILES].T.to_numpy(),
        index=data.group_profile_mix.index, columns=data.health_pivot.index,
    )
    observed_strength = {(r.group_id, r.food_id): float(r.behavior_strength) for r in data.behavior.itertuples(index=False)}
    groups = sorted(data.group_features.index)
    requests = sorted(data.similarity["request_food_id"].unique())
    rows = []
    model.eval()
    with torch.no_grad():
        states = model.encode(graph)
        for group_id in groups:
            group_idx = indices["household"][group_id]
            for request_food in requests:
                candidates = data.similarity[data.similarity["request_food_id"].eq(request_food)]
                if len(candidates) < 2:
                    continue
                for row in candidates.itertuples(index=False):
                    sub_food = row.substitute_food_id
                    pair_hf = torch.tensor([[group_idx, indices["food"][sub_food]]], dtype=torch.long, device=DEVICE)
                    pair_ff = torch.tensor([[indices["food"][request_food], indices["food"][sub_food]]], dtype=torch.long, device=DEVICE)
                    rows.append({
                        "group_id": group_id, "request_food_id": request_food, "substitute_food_id": sub_food,
                        "acceptance": float(model.predict_acceptance(states, pair_hf).item()),
                        "health": float(model.predict_health(states, pair_hf).item()),
                        "similarity": float(model.predict_similarity(states, pair_ff).item()),
                        "true_behavior": observed_strength.get((group_id, sub_food), 0.0),
                        "true_health": float(health_targets.loc[group_id, sub_food]),
                        "true_similarity": float(row.similarity_score),
                    })
    return pd.DataFrame(rows)


CANDIDATES = score_all_candidates(DATA, STANDARD_RESULT)
CANDIDATES.to_csv(OUTPUT_DIR / "tables" / "candidate_scores.csv", index=False)
print(f"scored {len(CANDIDATES)} candidate (group, request, substitute) rows")

scored 2142 candidate (group, request, substitute) rows


In [115]:
# ============================================================================
# 12. OBJECTIVE TRADE-OFF ANALYSIS  (replaces Pareto-frontier ranking)
#    Pareto dominance was removed per your request. In its place: a
#    continuous look at whether health and similarity actually trade off
#    against each other in this catalog, using correlation and a
#    quantile-binned trade-off curve — the same underlying multi-objective
#    question, without a discrete frontier-membership flag.
# ============================================================================
if len(CANDIDATES) > 0:
    corr = CANDIDATES[["acceptance", "health", "similarity"]].corr(method="spearman")
    corr.to_csv(OUTPUT_DIR / "tables" / "objective_correlations.csv")
    print("Spearman correlation between predicted objectives:")
    print(corr.round(3))

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    sc = axes[0].scatter(CANDIDATES["health"], CANDIDATES["similarity"], c=CANDIDATES["acceptance"],
                          cmap="viridis", alpha=0.6, s=18)
    axes[0].set_xlabel("Predicted health suitability")
    axes[0].set_ylabel("Predicted substitution similarity")
    axes[0].set_title("Objective trade-off space")
    fig.colorbar(sc, ax=axes[0], label="Predicted acceptance")

    binned = CANDIDATES.copy()
    binned["similarity_decile"] = pd.qcut(binned["similarity"], q=min(10, binned["similarity"].nunique()), duplicates="drop")
    tradeoff_curve = binned.groupby("similarity_decile", observed=True)["health"].mean()
    axes[1].plot(range(len(tradeoff_curve)), tradeoff_curve.to_numpy(), marker="o")
    axes[1].set_xlabel("Similarity decile (low -> high)")
    axes[1].set_ylabel("Mean predicted health suitability")
    axes[1].set_title("Health vs. similarity trade-off curve")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "figures" / "03_objective_tradeoff.png", dpi=600)
    plt.show()
    plt.close(fig)

Spearman correlation between predicted objectives:
            acceptance  health  similarity
acceptance       1.000  -0.168       0.088
health          -0.168   1.000      -0.168
similarity       0.088  -0.168       1.000


In [116]:
# ============================================================================
# 13. RELATION ABLATION STUDY  (new contribution — not in the original
#    paper). Retrains the standard-split model with each evidence channel
#    removed one at a time, quantifying how much behavior/health/similarity
#    performance depends on that relation type.
# ============================================================================
ablation_rows = []
full_metrics = STANDARD_RESULT.metrics
ablation_rows.append({
    "ablated_relation": "none (full model)", "behavior_roc_auc": full_metrics["behavior_roc_auc"],
    "health_mae": full_metrics["health_mae"], "similarity_mae": full_metrics["similarity_mae"],
})
ABLATION_EPOCHS = max(5, FINAL_EPOCHS // 3)
for relation_group in RELATION_GROUPS:
    print(f"ablation: retraining without '{relation_group}' relations ...")
    result = train_and_evaluate(
        DATA, STANDARD_FINAL, BEST_HP, seed=SEED + 999, epochs=ABLATION_EPOCHS,
        negative_ratio=BEST_NEGATIVE_RATIO, max_health_train=MAX_HEALTH_TRAIN // 2,
        max_health_test=MAX_HEALTH_TEST // 2, top_k=TOP_K, disabled_relations={relation_group},
    )
    ablation_rows.append({
        "ablated_relation": relation_group, "behavior_roc_auc": result.metrics["behavior_roc_auc"],
        "health_mae": result.metrics["health_mae"], "similarity_mae": result.metrics["similarity_mae"],
    })
ablation_df = pd.DataFrame(ablation_rows)
ablation_df["behavior_auc_drop"] = ablation_df["behavior_roc_auc"].iloc[0] - ablation_df["behavior_roc_auc"]
ablation_df["health_mae_increase"] = ablation_df["health_mae"] - ablation_df["health_mae"].iloc[0]
ablation_df.to_csv(OUTPUT_DIR / "tables" / "relation_ablation.csv", index=False)
print(ablation_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4.5))
plot_df = ablation_df[ablation_df["ablated_relation"] != "none (full model)"]
ax.bar(plot_df["ablated_relation"], plot_df["behavior_auc_drop"], color="tab:red", alpha=0.75)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("Behavior ROC AUC drop vs. full model")
ax.set_title("Relation Ablation: Which Evidence Channel Matters Most?")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "figures" / "04_relation_ablation.png", dpi=600)
plt.show()
plt.close(fig)

ablation: retraining without 'behavior' relations ...
ablation: retraining without 'nutrient' relations ...
ablation: retraining without 'category' relations ...
ablation: retraining without 'similarity' relations ...
 ablated_relation  behavior_roc_auc  health_mae  similarity_mae  behavior_auc_drop  health_mae_increase
none (full model)          0.947736    0.029001        0.037708           0.000000             0.000000
         behavior          0.800284    0.050811        0.051024           0.147453             0.021810
         nutrient          0.943178    0.064941        0.046482           0.004559             0.035939
         category          0.796788    0.042456        0.049213           0.150949             0.013455
       similarity          0.943614    0.041331        0.068739           0.004122             0.012329


In [117]:
# ============================================================================
# 14. BOOTSTRAP CONFIDENCE INTERVALS  (new contribution). The paper flagged
#    the 56-label similarity set as a real limitation; here we quantify
#    that uncertainty explicitly instead of reporting bare point estimates
#    for every split's test set.
# ============================================================================
def bootstrap_ci(y_true: np.ndarray, y_pred: np.ndarray, metric_fn, n_boot: int, seed: int) -> tuple:
    rng = np.random.default_rng(seed)
    n = len(y_true)
    values = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        try:
            values.append(metric_fn(y_true[idx], y_pred[idx]))
        except Exception:
            continue
    values = np.array([v for v in values if not math.isnan(v)])
    if len(values) == 0:
        return float("nan"), float("nan"), float("nan")
    return float(np.mean(values)), float(np.percentile(values, 2.5)), float(np.percentile(values, 97.5))


bootstrap_rows = []
for name, result in FINAL_RESULTS.items():
    b_auc = bootstrap_ci(result.test_behavior["binary_target"].to_numpy(dtype=float), result.behavior_scores,
                          lambda y, p: roc_auc_score(y, p) if len(np.unique(y)) > 1 else float("nan"),
                          N_BOOTSTRAP, SEED)
    h_mae = bootstrap_ci(result.test_health["target"].to_numpy(dtype=float), result.health_scores,
                          mean_absolute_error, N_BOOTSTRAP, SEED)
    if len(result.test_sim) > 0:
        s_mae = bootstrap_ci(result.test_sim["target"].to_numpy(dtype=float), result.sim_scores,
                              mean_absolute_error, N_BOOTSTRAP, SEED)
    else:
        s_mae = (float("nan"),) * 3
    for metric_name, (mean_v, lo, hi) in [("behavior_roc_auc", b_auc), ("health_mae", h_mae), ("similarity_mae", s_mae)]:
        bootstrap_rows.append({"split": name, "metric": metric_name, "mean": mean_v, "ci_lower": lo, "ci_upper": hi})
bootstrap_df = pd.DataFrame(bootstrap_rows)
bootstrap_df.to_csv(OUTPUT_DIR / "tables" / "bootstrap_confidence_intervals.csv", index=False)
print(bootstrap_df.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, metric_name in zip(axes, ["behavior_roc_auc", "health_mae", "similarity_mae"]):
    sub = bootstrap_df[bootstrap_df["metric"].eq(metric_name)]
    yerr = np.vstack([sub["mean"] - sub["ci_lower"], sub["ci_upper"] - sub["mean"]])
    ax.errorbar(range(len(sub)), sub["mean"], yerr=yerr, fmt="o", capsize=4, color="tab:blue")
    ax.set_xticks(range(len(sub)))
    ax.set_xticklabels([s.replace("_", " ").title() for s in sub["split"]], rotation=20, ha="right")
    ax.set_title(metric_name.replace("_", " "))
fig.suptitle(f"Bootstrap 95% Confidence Intervals (n={N_BOOTSTRAP} resamples)")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "figures" / "05_bootstrap_ci.png", dpi=600)
plt.show()
plt.close(fig)

           split           metric     mean  ci_lower  ci_upper
        standard behavior_roc_auc 0.947610  0.945535  0.949321
        standard       health_mae 0.029009  0.028660  0.029340
        standard   similarity_mae 0.037012  0.020825  0.054428
 cold_start_food behavior_roc_auc 0.749300  0.744571  0.754726
 cold_start_food       health_mae 0.071162  0.070467  0.071937
 cold_start_food   similarity_mae 0.125107  0.082807  0.173286
cold_start_group behavior_roc_auc 0.471204  0.465708  0.476655
cold_start_group       health_mae 0.025039  0.024681  0.025361
cold_start_group   similarity_mae 0.039465  0.022036  0.054947


In [118]:
# ============================================================================
# 15. BEHAVIOR-SCORE CALIBRATION  (new contribution). Behavior scores drive
#    ranking decisions directly; this checks whether a predicted 0.7 really
#    corresponds to ~70% observed positive behavior, via a reliability
#    diagram + Brier score, for the standard-split test set.
# ============================================================================
def reliability_diagram(y_true: np.ndarray, y_pred: np.ndarray, n_bins: int = 10) -> pd.DataFrame:
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(y_pred, bins[1:-1])
    rows = []
    for b in range(n_bins):
        mask = bin_ids == b
        if mask.sum() == 0:
            continue
        rows.append({
            "bin_center": (bins[b] + bins[b + 1]) / 2, "mean_predicted": float(y_pred[mask].mean()),
            "observed_rate": float(y_true[mask].mean()), "count": int(mask.sum()),
        })
    return pd.DataFrame(rows)


cal_df = reliability_diagram(
    STANDARD_RESULT.test_behavior["binary_target"].to_numpy(dtype=float), STANDARD_RESULT.behavior_scores
)
cal_df.to_csv(OUTPUT_DIR / "tables" / "behavior_calibration.csv", index=False)
brier = brier_score_loss(STANDARD_RESULT.test_behavior["binary_target"].to_numpy(dtype=float), STANDARD_RESULT.behavior_scores)
print(f"Behavior score Brier score (standard split test set): {brier:.4f}")

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfect calibration")
ax.plot(cal_df["mean_predicted"], cal_df["observed_rate"], marker="o", color="tab:blue", label="Model")
for _, row in cal_df.iterrows():
    ax.annotate(str(int(row["count"])), (row["mean_predicted"], row["observed_rate"]),
                fontsize=7, textcoords="offset points", xytext=(4, 4))
ax.set_xlabel("Mean predicted behavior score")
ax.set_ylabel("Observed positive rate")
ax.set_title(f"Behavior Score Calibration (Brier = {brier:.3f})")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "figures" / "06_calibration.png", dpi=600)
plt.show()
plt.close(fig)

Behavior score Brier score (standard split test set): 0.0888


In [119]:
# ============================================================================
# 16. COLD-START-GROUP FAIRNESS SEGMENTATION  (new contribution). The paper
#    flagged cold-start-group behavior AUC (0.42 in the original run) as a
#    real weakness. This checks *who* it fails hardest for: does it
#    generalize worse specifically for high-SNAP-share or low-income-band
#    household groups, i.e. exactly the populations food banks most need
#    to serve well?
# ============================================================================
cold_group_result = FINAL_RESULTS["cold_start_group"]
group_lookup = DATA.groups.set_index("group_id")[["poverty_band", "snap_band", "household_size_band"]]

seg = cold_group_result.test_behavior.copy()
seg["score"] = cold_group_result.behavior_scores
seg = seg.join(group_lookup, on="group_id")

segmentation_rows = []
for attribute in ["poverty_band", "snap_band", "household_size_band"]:
    for value, sub in seg.groupby(attribute):
        y = sub["binary_target"].to_numpy(dtype=float)
        p = sub["score"].to_numpy(dtype=float)
        auc = roc_auc_score(y, p) if len(np.unique(y)) > 1 else float("nan")
        segmentation_rows.append({"attribute": attribute, "value": value, "n": len(sub), "behavior_roc_auc": auc})
segmentation_df = pd.DataFrame(segmentation_rows)
segmentation_df.to_csv(OUTPUT_DIR / "tables" / "cold_start_group_fairness_segmentation.csv", index=False)
print(segmentation_df.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, attribute in zip(axes, ["poverty_band", "snap_band", "household_size_band"]):
    sub = segmentation_df[segmentation_df["attribute"].eq(attribute)]
    ax.bar(sub["value"].astype(str), sub["behavior_roc_auc"], color="tab:orange", alpha=0.8)
    ax.axhline(0.5, color="gray", linestyle="--", linewidth=0.8, label="Random (0.5)")
    ax.set_title(attribute.replace("_", " ").title())
    ax.set_ylabel("Behavior ROC AUC")
    ax.tick_params(axis="x", rotation=20)
fig.suptitle("Cold-Start-Group Generalization by Household Attribute")
axes[0].legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "figures" / "07_fairness_segmentation.png", dpi=600)
plt.show()
plt.close(fig)

          attribute              value     n  behavior_roc_auc
       poverty_band  above_two_poverty 18450          0.291974
       poverty_band      below_poverty 11215          0.711951
       poverty_band one_to_two_poverty 17603          0.577716
          snap_band           non_snap 30258          0.469279
          snap_band               snap 17010          0.443424
household_size_band              large 35232          0.532978
household_size_band             single  5515          0.457680
household_size_band              small  6521          0.170363


In [120]:
# ============================================================================
# 17. EMBEDDING-SPACE DIAGNOSTIC  (new contribution). PCA of the learned
#    food embeddings, colored by top-level category, plus a nearest-neighbor
#    sanity check: for a handful of sample foods, what does the model think
#    is most similar? This is an intuitive check on whether the GNN
#    organizes foods sensibly, independent of any single metric.
# ============================================================================
model, graph, indices = STANDARD_RESULT.model, STANDARD_RESULT.graph, STANDARD_RESULT.indices
model.eval()
with torch.no_grad():
    states = model.encode(graph)
    food_embeddings = states["food"].cpu().numpy()

food_ids_ordered = graph.node_ids["food"]
category_lookup = DATA.foods.set_index("food_id")["category_code"].to_dict()
categories = np.array([category_lookup.get(fid, -1) for fid in food_ids_ordered])

pca = PCA(n_components=2, random_state=SEED)
coords = pca.fit_transform(food_embeddings)
explained = pca.explained_variance_ratio_.sum()

fig, ax = plt.subplots(figsize=(7, 6))
sc = ax.scatter(coords[:, 0], coords[:, 1], c=categories, cmap="tab20", s=14, alpha=0.75)
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title(f"Learned Food Embedding Space (PCA, {explained:.0%} variance explained)\ncolored by food category code")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "figures" / "08_embedding_pca.png", dpi=600)
plt.show()
plt.close(fig)

sim_matrix = cosine_similarity(food_embeddings)
n_examples = min(5, len(food_ids_ordered))
rng_nn = np.random.default_rng(SEED)
sample_idx = rng_nn.choice(len(food_ids_ordered), size=n_examples, replace=False)
nn_rows = []
for idx in sample_idx:
    order = np.argsort(-sim_matrix[idx])
    order = [j for j in order if j != idx][:5]
    anchor_id = food_ids_ordered[idx]
    for rank, j in enumerate(order, start=1):
        nn_rows.append({
            "anchor_food_id": anchor_id, "anchor_description": DATA.food_descriptions.get(anchor_id, ""),
            "rank": rank, "neighbor_food_id": food_ids_ordered[j],
            "neighbor_description": DATA.food_descriptions.get(food_ids_ordered[j], ""),
            "cosine_similarity": float(sim_matrix[idx, j]),
        })
nn_df = pd.DataFrame(nn_rows)
nn_df.to_csv(OUTPUT_DIR / "tables" / "embedding_nearest_neighbors.csv", index=False)
print(nn_df.head(15).to_string(index=False))

          anchor_food_id                     anchor_description  rank          neighbor_food_id                                                 neighbor_description  cosine_similarity
FNDDS_2021_2023:27514040  Cheese steak sandwich or sub on wheat     1  FNDDS_2021_2023:27510657                     Double hamburger, on wheat bun, 2 medium patties           0.992970
FNDDS_2021_2023:27514040  Cheese steak sandwich or sub on wheat     2  FNDDS_2021_2023:27545110                               Turkey or chicken burger, on wheat bun           0.992725
FNDDS_2021_2023:27514040  Cheese steak sandwich or sub on wheat     3  FNDDS_2021_2023:54404000                                                 Popcorn chips, plain           0.991616
FNDDS_2021_2023:27514040  Cheese steak sandwich or sub on wheat     4  FNDDS_2021_2023:58102300                                                         Burrito, NFS           0.991389
FNDDS_2021_2023:27514040  Cheese steak sandwich or sub on wheat     5  FNDDS_202

In [121]:
# ============================================================================
# 18. PERMUTATION FEATURE IMPORTANCE  (new contribution). For the health
#    head: shuffle one food-feature column at a time and measure how much
#    MAE degrades, to see which nutrient/category signals the model
#    actually leans on rather than treating the GNN as a pure black box.
# ============================================================================
def permutation_importance_health(data: PreparedData, result: RunResult, n_repeats: int, seed: int) -> pd.DataFrame:
    model, graph, indices = result.model, result.graph, result.indices
    h_idx = make_index_tensor(result.test_health, "group_id", "food_id", indices["household"], indices["food"]).to(DEVICE)
    h_y = result.test_health["target"].to_numpy(dtype=float)
    rng = np.random.default_rng(seed)
    feature_names = list(data.food_features.columns)
    base_food_features = graph.node_features["food"].clone()

    def eval_mae(food_features_override):
        graph.node_features["food"] = food_features_override
        model.eval()
        with torch.no_grad():
            states = model.encode(graph)
            preds = model.predict_health(states, h_idx).cpu().numpy()
        return mean_absolute_error(h_y, preds)

    base_mae = eval_mae(base_food_features)
    rows = []
    n_features = base_food_features.shape[1]
    for f in range(n_features):
        degradations = []
        for _ in range(n_repeats):
            perturbed = base_food_features.clone()
            perm = torch.tensor(rng.permutation(perturbed.shape[0]), device=perturbed.device)
            perturbed[:, f] = perturbed[perm, f]
            degradations.append(eval_mae(perturbed) - base_mae)
        rows.append({"feature": feature_names[f], "mean_mae_increase": float(np.mean(degradations)),
                      "std_mae_increase": float(np.std(degradations))})
    graph.node_features["food"] = base_food_features
    return pd.DataFrame(rows).sort_values("mean_mae_increase", ascending=False)


PERM_REPEATS = 2 if QUICK_TEST_MODE else 5
importance_df = permutation_importance_health(DATA, STANDARD_RESULT, PERM_REPEATS, SEED)
importance_df.to_csv(OUTPUT_DIR / "tables" / "permutation_feature_importance.csv", index=False)
print(importance_df.head(15).to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 6))
top_features = importance_df.head(15)
ax.barh(top_features["feature"][::-1], top_features["mean_mae_increase"][::-1],
        xerr=top_features["std_mae_increase"][::-1], color="tab:green", alpha=0.8)
ax.set_xlabel("Health-MAE increase when feature is shuffled")
ax.set_title("Permutation Feature Importance (health prediction head)")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "figures" / "09_permutation_importance.png", dpi=600)
plt.show()
plt.close(fig)

                       feature  mean_mae_increase  std_mae_increase
                       fiber_g           0.011375          0.000140
                  potassium_mg           0.008251          0.000230
                     protein_g           0.005884          0.000168
                     sodium_mg           0.005396          0.000090
                carbohydrate_g           0.004857          0.000087
                    calcium_mg           0.004565          0.000091
                total_sugars_g           0.003795          0.000160
               saturated_fat_g           0.003010          0.000060
                       iron_mg           0.002192          0.000046
          missing_vitamin_c_mg           0.001858          0.000058
                   total_fat_g           0.001516          0.000042
               missing_fiber_g           0.001453          0.000027
source_release_FNDDS_2011_2012           0.000780          0.000047
                  vitamin_c_mg           0.00066

In [122]:
# ============================================================================
# 19. MANIFEST + FINAL SUMMARY
# ============================================================================
manifest = {
    "experiment": "source-grounded-graph-food-substitution (Kaggle edition)",
    "device": str(DEVICE),
    "quick_test_mode": QUICK_TEST_MODE,
    "database": str(DB_PATH), "catalog": str(CATALOG_PATH), "substitution_pairs": str(PAIRS_PATH),
    "best_hyperparameters": {**BEST_HP, "negative_ratio": BEST_NEGATIVE_RATIO},
    "optuna": {"n_trials": N_OPTUNA_TRIALS, "best_value": study.best_value},
    "table_counts": DATA.table_counts,
    "mapped_similarity_labels": int(len(DATA.similarity)),
    "final_metrics": metrics_frame.to_dict(orient="records"),
    "relation_ablation": ablation_df.to_dict(orient="records"),
    "removed_from_original": ["pareto_frontier ranking / frontier-membership diagnostic / pareto_coverage metric"],
    "new_analyses": [
        "objective trade-off correlation + quantile trade-off curve (replaces Pareto)",
        "relation ablation study", "bootstrap confidence intervals",
        "behavior-score calibration (reliability diagram + Brier score)",
        "cold-start-group fairness segmentation by SNAP/poverty/household-size band",
        "embedding-space PCA + nearest-neighbor sanity check",
        "permutation feature importance for the health head",
        "proper train/val/test split (was train/test with an unused 10% gap)",
        "Bayesian architecture search over width/depth/dropout/residual/activation/aggregation/head-size",
    ],
}
(OUTPUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2, default=str), encoding="utf-8")

print("\n" + "=" * 70)
print("DONE. Outputs written to:", OUTPUT_DIR)
print("=" * 70)
print(f"Best Optuna trial value : {study.best_value:.4f}  ({N_OPTUNA_TRIALS} trials)")
print(f"Best architecture       : {json.dumps(BEST_HP, indent=2)}")
print("\nFinal metrics:")
print(metrics_frame[["split", "behavior_roc_auc", "behavior_ndcg_at_5", "health_mae",
                      "health_spearman", "similarity_mae", "similarity_spearman"]].to_string(index=False))
print(f"\nFigures  -> {OUTPUT_DIR / 'figures'}")
print(f"Tables   -> {OUTPUT_DIR / 'tables'}")
print(f"Checkpoints -> {OUTPUT_DIR / 'checkpoints'}")


DONE. Outputs written to: /kaggle/working
Best Optuna trial value : 1.8714  (30 trials)
Best architecture       : {
  "hidden_dim": 64,
  "layers": 2,
  "dropout": 0.21768432891642056,
  "residual": true,
  "activation": "relu",
  "aggregation": "sum",
  "head_hidden_ratio": 2,
  "lr": 0.003480961488805864,
  "weight_decay": 1.338493154542602e-06,
  "w_acceptance": 1.9055752192176885,
  "w_health": 1.3353701724932696,
  "w_similarity": 1.0828257706200823
}

Final metrics:
           split  behavior_roc_auc  behavior_ndcg_at_5  health_mae  health_spearman  similarity_mae  similarity_spearman
        standard          0.947736            0.996876    0.029001         0.939289        0.037708             0.693842
 cold_start_food          0.749144            0.483525    0.071144         0.898379        0.126249             0.665398
cold_start_group          0.471291            0.250000    0.025030         0.939715        0.039479             0.692618

Figures  -> /kaggle/working/figures
T